## 🧠 Building an Agent with LangGraph + DSPy

In Part 1, we built a powerful **multimodal agent system** using no-code tools.

Now, we go one level deeper.

#### What is different in this notebook?

We will rebuild the same system using:

- **LangGraph** → to define the agent workflow  
- **DSPy** → to control how each node *thinks*  

👉 Instead of “configuring” an agent, we now **design it step by step**.

### 🧩 Workshop Approach

For each node in the system:

1. We explain its **role**
2. You write the **prompt (DSPy Signature)**
3. You can use a **hint if needed**
4. You test the node in isolation

### 📦 Setup

We install all required libraries and configure the tools needed.

In [0]:
%pip install langgraph==1.0.4 langchain-core==1.1.3 langchain-openai==1.1.1 databricks-ai-bridge==0.9.0 langchain==1.1.3 pydantic==2.12.4 mlflow==3.6.0 dspy-ai==3.1.3
%restart_python

In [0]:
%run ../Setup/setup__

In [0]:
import os
from typing import Annotated, Optional, TypedDict

import dspy
import mlflow.deployments
import mlflow
from databricks.sdk import WorkspaceClient
from databricks_ai_bridge.genie import Genie
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

import warnings

warnings.filterwarnings("ignore")

### ⚙️ Configuration

Before running the agent, we need to connect it to the tools we created in Part 1.

---

### 🔑 What are these IDs?

#### Genie Space ID
This connects your agent to the **structured financial data (tables)**.

📍 Where to find it:
- Go to **Genie**
- Open your space
- In the **About** section copy the **Space ID**

---

#### Knowledge Assistant Endpoint
This connects your agent to the **document-based knowledge (PDF / RAG)**.

📍 Where to find it:
- Go to **Agents**
- Find your Knowledge Assistant (sds-ka-yourname)
- Click the **Endpoint** button with the green light
- Copy the **Endpoint name**

In [0]:
# Initialize Databricks SDK
w = WorkspaceClient()

# Configure DSPy to use Databricks-hosted Llama 3.3
# No API keys needed when running inside a Databricks Notebook
lm = dspy.LM(model="databricks/databricks-meta-llama-3-3-70b-instruct")
dspy.settings.configure(lm=lm)

### ✍️ Put here the IDs for the tools ###
GENIE_SPACE_ID = ""
KA_ENDPOINT_NAME = ""

### 🗂️ What are we defining here?

#### 📝 Conversation History
We store past interactions so the agent can:
- Remember previous questions
- Handle follow-ups naturally

👉 Example:
> "What about 2025?" → the agent knows what “that” refers to

---

#### 🔄 Agent State
This is the **information that flows between steps** of the agent.

Each node (step) adds something:
- A rewritten question
- A tool result
- A final answer

---

#### 🧾 Structured Outputs (Important!)

We define **clear formats** for certain steps:

- **Normalizer Output** → rewritten query + intent classification  
- **Supervisor Output** → decision (is the answer complete?)

💡 Why this matters:
Instead of vague responses, the agent produces **structured, reliable outputs**  
→ This makes the workflow more predictable and easier to debug

---

🚫 No action needed here  
✅ Just run the cell — this sets up the foundation for the agent

In [0]:
# A single turn in the conversation history.
class HistoryEntry(BaseModel):
    query: str
    answer: str
    tool: str | None = None

# This is what the Agent saves in every turn
class AgentState(TypedDict):
    query: str
    messages: Annotated[list, add_messages]
    normalized_query: str
    tool_choice: str # 'genie', 'ka', 'chitchat', 'fallback'
    genie_output: Optional[str]
    ka_output: Optional[str]
    final_answer: Optional[str]
    history: list[HistoryEntry]

# A structured output for the normalizer
class NormalizerResult(BaseModel):
    normalized_query: str = Field(description="The standalone rewritten query")
    tool_hint: str = Field(description="One of: 'informative', 'chitchat', 'fallback'")

# A structured output for the supervisor
class SupervisorOutput(BaseModel):
    is_complete: bool = Field(description="True if the genie_output fully answers the query, False if more info is needed.")
    reason: str = Field(description="Brief reason for the decision.")

### 🏗️ Agent Architecture Overview

Before we build each component, let’s understand the full workflow.

---

### 🔀 High-Level Flow

Our agent follows this decision process:

1. 🧩 **Normalize the query**
2. 🚦 **Classify intent**
   - 💬 chitchat → respond directly  
   - ❌ fallback → reject  
   - 📊 informative → continue  
3. 📊 **Query structured data (Genie)**
4. 🧠 **Decide if answer is complete**
5. 📄 (Optional) **If not, query documents (Knowledge Assistant)**
6. ✨ **Refine final answer**
7. 🧾 **Store conversation history**

We will define each of them later in more detail.

---

### 🤔 Why this architecture?

This is **not the only way** to build an agent — it is an **intentional design choice**.

1. Calling the Genie first is **time efficient** and **low-cost**.
2. Using a **smart decision layer** avoids unnecessary **costs** and **slow responses**.
3. Merging and refining the tool outputs gives us **freedom** in the format of the **final response**.
4. Each tool has its own specific purpose, making errors easier to **debug**.

👉 In a real-world scenario different architectures should be created and **evaluated**.

### 🧾 Conversation Memory

Before building the agent nodes, we define how the agent **stores and uses conversation history**.

#### 🔧 What we implement

We create two helper functions:

1. **Format history for prompts** → so LLMs can understand past conversation  
2. **Update history** → so the agent remembers each interaction  

---

⚠️ You do NOT need to modify this code — it is used by multiple nodes internally.

In [0]:
def format_history_for_prompt(history: list[dict[str, str]] | None) -> str:
    """Format the conversation history for inclusion in a prompt."""
    if not history:
        return "No previous conversation."

    blocks = [
        f"User asked: {entry['query']}\nAssistant answered: {entry['answer']}"
        for entry in history
    ]

    return "\n\n".join(blocks)


def update_history_node(state: dict) -> dict:
    """Update the conversation history with the latest interaction."""
    history = state.get("history") or []
    ans = state.get("final_answer") or ""

    new_entry = HistoryEntry(
        query=state.get("query"),
        answer=ans,
        tool=state.get("tool_choice")
    )

    history.append(new_entry)

    return {"history": history[-5:]}

### 🧩 Normalizer node

#### 🎯 Objective

This node has TWO responsibilities:
1. Rewrite the query so it is **clear and standalone**
2. Classify the query into:
   - `informative` → financial / business question
   - `chitchat` → greetings / casual conversation
   - `fallback` → out-of-scope questions


#### ✍️ Your Task

Write instructions that:

- Use conversation history when needed  
- Rewrite vague queries
- Correctly classify intent, even when unclear

💡 Tip:
This is a **guided prompt** to help you get started. In later nodes, you will write prompts fully from scratch.

In [0]:
class NormalizerSignature(dspy.Signature):
    """ Rewrite the query into a _______ question if it's vague.
    Use the _______ if needed. 
    Also assign a tool category.

    Tool categories:
    - _______: For general chat, greetings, etc.
    - _______: For queries about Avolta's financial performance and metrics.
    - _______: For queries that are out-of-scope, like general knowledge.

    If you're not sure default to "_______" as a tool hint."""
    history = dspy.InputField()
    user_input = dspy.InputField()
    output: NormalizerResult = dspy.OutputField()

def normalizer_node(state):
    predictor = dspy.Predict(NormalizerSignature)
    result = predictor(history=format_history_for_prompt(state.get("history", [])),
                       user_input=state["query"])
    return {
        "normalized_query": result.output.normalized_query,
        "selected_tool": result.output.tool_hint
    }

In [0]:
load_hint("normalizer")

🧪 In the following cell you can test how well your prompt is performing.

The given query is poorly written but should fall into "**informative**".

👉 Does the normalizer transform it to a standalone clear question?

Try to put queries you expect to be in all different categories. 

In [0]:
### TEST THE NORMALIZER NODE ###
query = "what is avolta's core proftt in 20224?" # Try to put different queries

test_state = {"query": query, "history": []}

output = normalizer_node(test_state)

print(f"Normalized Query: {output['normalized_query']}")
print(f"Selected Tool: {output['selected_tool']}")

### 💬 Chitchat & Fallback Nodes

These nodes handle queries that do NOT require financial reasoning.

---

### Chitchat Node

This node is responsible for:
- Greetings
- Small talk
- Casual conversation

It should respond in a friendly, natural, and professional tone.

---

### Fallback Node

This node is triggered when the query is **out of scope**.

It should:
- Politely refuse
- Clearly state the system's limitations
- Avoid attempting to answer unrelated questions

---

### ✍️ Your Task

For the chitchat node:
- Define how the assistant should respond to greetings and casual inputs

For the fallback node:
- Define a deafult answer suitable if the query is out-of-scope

In [0]:
class ChitChatSignature(dspy.Signature):
    """
    ✍️ Write your chitchat prompt here
    """
    user_input = dspy.InputField()
    answer = dspy.OutputField()

def chitchat_node(state):
    chat = dspy.Predict(ChitChatSignature)
    result = chat(user_input=state["query"])
    return {"final_answer": result.answer}

In [0]:
load_hint("chitchat")

In [0]:
def fallback_node(state):
    fallback_answer = """✍️ Write a default answer that will be given to fallback questions."""
    return {"final_answer": fallback_answer}

In [0]:
load_hint("fallback")

🧪 In the following cell you can test how well your chitchat prompt is performing.

👉 Does the tool answer in the expected manner?

In [0]:
### TEST CHITCHAT NODE ###
query = "hello, how are you?" # try to put different small talk queries here

ans = chitchat_node({"query": query})
print(ans)

### 🔎 Genie node

The Genie node queries a structured data source (via a Genie agent) to retrieve precise, data-driven answers. It performs text-to-SQL transformation.

⚠️ You do NOT need to modify this code. Run it and examine the results.

In [0]:
def genie_node(state):
    query_text = state["normalized_query"]

    # Initialize Genie and ask a question
    genie_agent = Genie(GENIE_SPACE_ID)
    genie_ans = genie_agent.ask_question(query_text)

    return {"genie_output": genie_ans.result if genie_ans else "Sorry, no response."}

In [0]:
### TEST THE GENIE NODE ###
query = "What was the Net Profit for the half year ended June 30, 2025?"

test_state = {"normalized_query": query}
genie_ans = genie_node(test_state)
print(genie_ans['genie_output']) #expected 88 million CHF

### 🧠 Supervisor Decision Node

#### 🎯 Objective

This node acts as the **decision-making brain** of the system.

Its goal is to evaluate whether the **Genie output is sufficient** to answer the user’s query, or if additional reasoning and context are required.

It decides the next step in the workflow:
- Use the current Genie answer as final
- OR route the query to the Knowledge Assistant for deeper analysis

#### ✍️ Your Task

Write instructions that help the model:

- Judge whether the answer is **complete or incomplete** so it decides the next step

In [0]:
class SupervisorSignature(dspy.Signature):
    """
    ✍️ Write your supervisor prompt here
    """
    query = dspy.InputField(desc="The user's query.")
    genie_output = dspy.InputField(desc="The Genie answer.")
    output: SupervisorOutput = dspy.OutputField()

def supervisor_node(state):
    decider = dspy.Predict(SupervisorSignature)
    result = decider(query=state["query"], genie_output=state.get("genie_output", ""))

    is_complete_bool = result.output.is_complete
    
    # Logic will now work correctly
    next_step = "refiner" if is_complete_bool else "ka_node"
    return {"next_node": next_step}

In [0]:
load_hint("supervisor")

🧪 In the following cell you can test how well your prompt is performing.

👉 The output of the Genie doesnt't answer the test query, so next node should be the knowledge assistant.

You can try to put different queries and answers and see how strict the supervisor is.

In [0]:
### TEST THE SUPERVISOR NODE ###
query = "Why did revenue increase?" # here you can put a different query
genie_output = "Revenue was 6,734 million CHF." # here you can put a different genie answer from genie

test_state = {"query": query, "genie_output": genie_output}
next_node = supervisor_node(test_state)['next_node']

print("next_node :", next_node) # expected: ka_node

### 📚 Knowledge Assistant node

The Knowledge Assistant node uses a language model endpoint to retrieve and synthesize information from unstructured knowledge sources if it is decided by the supervisor that further research is needed.

⚠️ You do NOT need to modify this code. Run it and examine the results.

In [0]:
def ka_node(state):
    query_text = state["normalized_query"]

    client = mlflow.deployments.get_deploy_client("databricks")
    
    # Query the Agent serving endpoint directly
    response = client.predict(
        endpoint=KA_ENDPOINT_NAME,
        inputs={
            "input": [{"role": "user", "content": query_text}]
        }
    )
    
    # Extract the final answer from the Knowledge Assistant trace
    ka_text = response.output[0]['content'][0]['text']
    
    return {"ka_output": ka_text}

In [0]:
### TEST THE KNOWLEDGE ASSISTANT NODE ###
query = "What company did Dufry combine with on February 3, 2023, to form the Avolta Group?"

test_state = {"normalized_query": query}
ka_ans = ka_node(test_state)

print(ka_ans['ka_output'])  # expected: Autogrill

### ✨ Refiner node

This merges the Genie and the Knowledge Assistant outputs into a final executive response.

#### 🎯 Objective

The Refiner is where the magic happens. It takes the two answers and blends them into a final answer. 

**Crucial Rule:** The user should never know we used two different tools. It should feel like the assistant "just knows" the answer.

#### ✍️ Your Task

Write instructions that:

- Merge Genie data and Knowledge assistant context into a single response
- Do not  mention which tool answered what
- If neither tool answers, the agent shouldn't hallucinate
- Keep the tone professional

In [0]:
class RefinerSignature(dspy.Signature):
    """
    ✍️ Write your refiner prompt here
    """
    query = dspy.InputField()
    genie_content = dspy.InputField()
    ka_content = dspy.InputField()
    refined_answer = dspy.OutputField()

def refiner_node(state):
    editor = dspy.ChainOfThought(RefinerSignature)
    result = editor(
        query=state["query"],
        genie_content=state.get("genie_output", "No Genie data found."),
        ka_content=state.get("ka_output", "No Knowledge Assistant context found.")
    )
    return {"final_answer": result.refined_answer}

In [0]:
load_hint("refiner")

🧪 In the following cell you can test how well your refiner prompt is performing.

👉 Watch how the refiner answers when one tool has answered clearly. Does it include the additional information from the other tool?

👉 Watch how the refiner handles no answers from both tools. Does it hallucinate?

In [0]:
### TEST THE REFINER NODE ###
test_state = {"query": "What is the revenue for the year ended December 31, 2024?",
              "genie_output": "The revenue of 2024 is 6.7B",
              "ka_output": "Most of the 2024 revenue is due to tourism."}

refiner_ans = refiner_node(test_state)['final_answer']
print("answer :", refiner_ans)

In [0]:
# --- TEST THIS NODE ---
test_state = {"query": "How many treasury shares did Avolta cancel in December 2024?",
              "genie_output": "There is no information about that.",
              "ka_output": "Avolta cancelled 4,230 thousand shares in May 2022."}

refiner_ans = refiner_node(test_state)['final_answer']
print("answer :", refiner_ans)

### 🔗 Building the Agent Graph

Now that we’ve defined all the individual nodes, it’s time to **connect everything into a working system**.

We define:
- **How the agent flows**
- **Which node runs next**
- **How decisions are made dynamically**

---

#### 🔀 Conditional Routing Logic

This graph is **dynamic**, not linear.

Two key decision points:

- **After Normalizer → where do we go?**
- **After Genie → do we stop or dig deeper?**

This is what makes the agent *agentic* rather than a simple pipeline.

---
### ✍️ What you should do now

👉 Simply **run the cell below** to assemble your agent and then visualize it.

At this stage:
- You are **not writing new logic**
- You are **connecting everything you already built**

In [0]:
def route_after_normalization(state):
    """Routes to Genie, Chitchat, or Fallback based on user intent."""
    return state["selected_tool"]

def route_after_supervisor(state):
    """The 'Supervisor' logic. Decides if we need the PDF (KA) or we are done."""
    # We call your supervisor node logic here
    # It must return the STRING of the next node
    decision = supervisor_node(state) 
    return decision["next_node"]

def get_compiled_graph() -> StateGraph:
    workflow = StateGraph(AgentState)

    # Add Nodes
    workflow.add_node("normalizer", normalizer_node)
    workflow.add_node("genie_node", genie_node)
    workflow.add_node("ka_node", ka_node)
    workflow.add_node("refiner", refiner_node)
    workflow.add_node("chitchat", chitchat_node)
    workflow.add_node("fallback", fallback_node)
    workflow.add_node("update_history", update_history_node)

    # Start at Normalizer
    workflow.add_edge(START, "normalizer")

    # Conditional: Normalizer -> (Genie, Chitchat, or Fallback)
    workflow.add_conditional_edges(
        "normalizer",
        route_after_normalization,
        {
            "informative": "genie_node",
            "chitchat": "chitchat",
            "fallback": "fallback"
        }
    )

    # Conditional: Genie -> (Refiner OR Knowledge Assistant)
    workflow.add_conditional_edges(
        "genie_node",
        route_after_supervisor,
        {
            "refiner": "refiner",
            "ka_node": "ka_node"
        }
    )

    # Add edges
    workflow.add_edge("ka_node", "refiner")
    workflow.add_edge("refiner", "update_history")
    workflow.add_edge("chitchat", "update_history")
    workflow.add_edge("fallback", "update_history")
    workflow.add_edge("update_history", END)

    app = workflow.compile()

    return app

In [0]:
app = get_compiled_graph()
app

### 🚀 Running the Agent

We wrap everything into a simple `ask()` function so users can interact with the full system like a real assistant.

- Input: a user question  
- Output: a final answer + updated memory

👉 Simply **run the cell below** to create the ask function that will invoke your agent.

In [0]:
def ask(query: str, history: list = None):
    """
    Invokes the Avolta Financial Agent.
    Returns: The final text answer and the updated history.
    """
    if history is None:
        history = []

    # Initial state for the graph
    initial_state = {
        "query": query,
        "history": history,
        "messages": [] # LangGraph internal message tracking
    }

    # Run the graph
    result = app.invoke(initial_state)

    # Extract the answer from whichever node finished last
    final_answer = result.get("final_answer", "I'm sorry, I couldn't generate an answer.")
    
    # Update the history
    history.append({"query": query, "answer": final_answer})

    return final_answer, history

#### 1. 🤔 This question should be answered using only Genie

In [0]:
answer, updated_history = ask("What was the Net profit for the half year ended June 30, 2025?")
print(answer)

#### 2. 🤔 This question should be answered using the Knowledge Assistant

In [0]:
### Test the Genie to see it cannot answer ###
query = "What is Avolta's medium-term leverage target?"

test_state = {"normalized_query": query}
genie_ans = genie_node(test_state)
print(genie_ans['genie_output'])

In [0]:
answer, updated_history = ask("What is Avolta's medium-term leverage target?")
print(answer)

#### ⛔ This is not a question the agent can answer

In [0]:
answer, updated_history = ask("what is the weather like today?")
print(answer)

#### 👋 This is just small talk.

In [0]:
answer, updated_history = ask("ok thank you, goodbye")
print(answer)

## 🚀 From Prototype to Production

So far, we have built a fully functional **multi-agent system** using LangGraph.

- ✅ We designed the architecture  
- ✅ We implemented each node (Normalizer, Supervisor, etc.)  
- ✅ We tested the agent interactively in the notebook  

But there is one big limitation:

> ❗ Right now, our agent only lives inside this notebook.

---

### 🧠 Why Production Matters

In real-world applications, we need our agent to be:

- 🌐 Accessible via APIs  
- ⚡ Scalable to multiple users  
- 🔁 Reusable across applications (apps, dashboards, copilots)  
- 🔒 Governed and secure  

---

### 🏗️ What We’ll Do Next

We will take our notebook-based agent and:

1. 📦 Wrap it into an MLflow-compatible format  
2. 🏷️ Register it in Unity Catalog  
3. 🚀 Deploy it as a real-time serving endpoint
---

⚠️ You don't need to change anything on the cells below, just run them to deploy your agent.

### 📦 Wrapping the Agent for Production

We now take our LangGraph agent and wrap it into a **production-ready interface** using MLflow.

👉 So that Databricks can:
- Serve the agent behind an API endpoint  
- Handle requests from the Playground or external apps  
- Support streaming responses (like ChatGPT-style output)

---

#### ⚙️ Key Components

**1. `load_context()` → Initialization**
- Runs once when the endpoint starts
- Loads the LLM (via DSPy)
- Compiles the LangGraph
- Ensures everything is ready before serving traffic

---

**2. `predict_stream()` → Core Logic 🚀**
- This is the **heart of the deployed agent**
- It:
  - Extracts the user query + conversation history  
  - Invokes the LangGraph agent  
  - Streams the response back

---

**3. `predict()` → Fallback**
- Wraps the streaming output into a single response
- Used when streaming is not supported

---

In [0]:
from mlflow.pyfunc import ResponsesAgent 
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from typing import Generator

class AvoltaAgent(ResponsesAgent):
    def __init__(self) -> None:
        """Initialize the AvoltaAgent with the provided agent."""
        self.agent = None

    def load_context(self, context: dict) -> None:
        """Run once when the serving endpoint starts up."""
        import dspy

        lm = dspy.LM(model="databricks/databricks-meta-llama-3-3-70b-instruct")
        dspy.settings.configure(lm=lm)

        self.agent = get_compiled_graph()

        if self.agent is None:
            raise RuntimeError("Failed to compile LangGraph. check get_compiled_graph() logic.")
    
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        """Non-streaming fallback: Collects all stream events into one response."""
        items = []
        for event in self.predict_stream(request):
            e_type = event.get("type") if isinstance(event, dict) else getattr(event, "type", None)
            
            if e_type == "response.output_item.done":
                e_item = event.get("item") if isinstance(event, dict) else getattr(event, "item", None)
                items.append(e_item)

        return ResponsesAgentResponse(output=items)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """The core logic that powers the Playground UI with streaming."""
        user_query = ""
        history = []

        # Reconstruct Query and History from the 'input' field
        if request.input:
            # The last message is the current query
            last_msg = request.input[-1]
            user_query = last_msg.get("content", "") if isinstance(last_msg, dict) else getattr(last_msg, "content", "")

            # Previous messages are the history (User, Assistant, User, Assistant...)
            prev_messages = request.input[:-1]
            for i in range(0, len(prev_messages) - 1, 2):
                u_msg = prev_messages[i]
                a_msg = prev_messages[i+1]
                
                q = u_msg.get("content", "") if isinstance(u_msg, dict) else getattr(u_msg, "content", "")
                a = a_msg.get("content", "") if isinstance(a_msg, dict) else getattr(a_msg, "content", "")
                
                if q and a:
                    history.append({"query": str(q), "answer": str(a)})

        # Prepare the LangGraph Initial State
        initial_state = {
            "query": user_query,
            "history": history,
            "messages": [{"role": "user", "content": user_query}]
        }

        item_id = "assistant_response"

        try:
            # Invoke the LangGraph
            final_state = self.agent.invoke(initial_state)
            final_text = final_state.get("final_answer") or "I'm sorry, I couldn't find an answer."

            # Yield the text chunks for the UI
            words = final_text.split(" ")
            for i, word in enumerate(words):
                delta = word + (" " if i < len(words) - 1 else "")
                yield self.create_text_delta(delta=delta, item_id=item_id)

            # Signal that the response is complete
            final_item = self.create_text_output_item(text=final_text, id=item_id)
            yield ResponsesAgentStreamEvent(type="response.output_item.done", item=final_item)

        except Exception as e:
            error_text = f"Agent Error: {str(e)}"
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_text_output_item(text=error_text, id="error")
            )

### 🔗 Declaring Dependencies & Logging the Model

Before deploying, we must tell Databricks:

👉 “What does this agent depend on?”

This ensures the agent runs reliably in production.

---

#### 📦 Resource Dependencies

We explicitly declare all external resources the agent uses:

- 🧠 **Genie Space** → for structured data queries  
- 📚 **Knowledge Assistant Endpoint** → for PDF reasoning  
- 📊 **Unity Catalog Tables** → financial datasets  

---

#### 💡 Requirements

`pip_requirements`: We pin the exact versions of dspy, langgraph, and pydantic. 

This prevents the "it worked on my machine" problem by recreating your environment exactly on the serving cluster.

---

#### ⚙️ What gets logged?

- The **agent class (`AvoltaAgent`)**
- All **Python dependencies**
- The **resource graph**
- An **input example** (for testing & schema)



### ✍️ What you should do
Complete the unique user id as described in the comment and run the cell

In [0]:
from mlflow.models.resources import (
    DatabricksGenieSpace,
    DatabricksSQLWarehouse,
    DatabricksServingEndpoint,
    DatabricksTable
)

### PUT YOUR FIRST NAME AND PART OF YOUR SURNAME (e.g. georgetz) HERE
unique_user_id = ""

# Define your Resource dependencies
resources = [
    DatabricksGenieSpace(genie_space_id=GENIE_SPACE_ID),
    DatabricksServingEndpoint(endpoint_name=KA_ENDPOINT_NAME),
    DatabricksTable(table_name="sds.cleaned.balance_sheet_2024"),
    DatabricksTable(table_name="sds.cleaned.cash_flow_2024"),
    DatabricksTable(table_name="sds.cleaned.core_ifrs_2024"),
    DatabricksTable(table_name="sds.cleaned.equity_free_cash_flow_2024"),
    DatabricksTable(table_name="sds.cleaned.income_statement_2024"),
    DatabricksTable(table_name="sds.cleaned.balance_sheet_2025_6m"),
    DatabricksTable(table_name="sds.cleaned.cash_flow_2025_6m"),
    DatabricksTable(table_name="sds.cleaned.core_ifrs_2025_6m"),
    DatabricksTable(table_name="sds.cleaned.equity_free_cash_flow_2025_6m"),
    DatabricksTable(table_name="sds.cleaned.income_statement_2025_6m"),
    DatabricksSQLWarehouse(warehouse_id="9970bd5c622f474a")
]

mlflow.set_experiment(f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/avolta_agent")

with mlflow.start_run(run_name="avolta_agent_traces"):
    model_info = mlflow.pyfunc.log_model(
        python_model=AvoltaAgent(),
        artifact_path="agent",
        resources=resources,
        pip_requirements=[
            "databricks-agents==1.8.1",
            "databricks-sdk==0.67.0",
            "databricks-ai-bridge==0.9.0",
            "pydantic==2.12.4",
            "langgraph==1.0.4",
            "langchain-core==1.1.3",
            "langchain-openai==1.1.1",
            "langchain==1.1.3",
            "dspy-ai==3.1.3"
        ],
        input_example={"input": [{"role": "user", "content": "What is the revenue?"}]}
    )

### 🏷️ Step 3: Registering the Model in Unity Catalog

Now we take the logged MLflow model and **register it in Unity Catalog**.

---

#### 📚 What is happening?

We assign a permanent name:

` sds.default.avolta_finance_agent `

This creates:
- A **versioned model registry**
- A central place to manage deployments

---

#### 🔢 Why versioning matters

Each time you improve your agent:
- A new version is created  
- You can compare, rollback, or promote versions  

In [0]:
UC_MODEL_NAME = f"sds.default.avolta_agent_{unique_user_id}"

# Register the model
model_version = mlflow.register_model(
    model_uri=f"runs:/{model_info.run_id}/agent",
    name=UC_MODEL_NAME
)

### 🚀 Step 4: Deploying the Agent

This is the final step — we deploy the agent as a **real-time API endpoint**.

---

#### ⚙️ What happens here?

- Databricks creates a **serving endpoint**
- Your agent is now accessible via:
  - API calls  
  - Playground UI
  - Review App

---

#### 💡 Key Feature: `scale_to_zero`

- The endpoint **automatically shuts down when idle** to save cost 💰
- Spins back up when a request comes in

---

#### 🌐 What do you get?

After deployment:

👉 A link to the **AI Playground**

There you can chat with your agent  

---


---

💡 Try asking:
- Financial questions 📊  
- Analytical “why” questions 📚  
- Even edge cases to test robustness 🧠  

In [0]:
from databricks import agents

# Deploy the version we just registered
deployment = agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=model_version.version,
    endpoint_name="avolta_agent_"+unique_user_id,
    scale_to_zero=True
)

print(f"✅ Deployment started!")
print(f"👉 Open the AI Playground here: {deployment.review_app_url}")

### 🎉 Final Result

You now have a:
- Multi-tool agentic system  
- With structured + unstructured reasoning  
- Fully deployed as a scalable endpoint  